# Document Q&A with Multi-Provider RAG

Ask questions over your own PDFs, text, or Markdown files. Answers are grounded only in the document you upload. No data is written to disk, everything runs in memory for the session.

**What this does:** Upload a doc → get answers that cite only that doc. You can switch between OpenRouter (e.g. GPT-4o-mini) and Ollama (local) for both the LLM and embeddings. Index and vector store are in-memory only and disappear when you stop the app.

#### Pipeline (high level)

1. **Ingest** — Upload is checked (max size 5 MB), then split into overlapping chunks (800 chars, 150 overlap).
2. **Index** — Chunks are embedded with the chosen provider and stored in an in-memory Chroma collection.
3. **Query** — Your question is run against the store; we pull the top-k chunks via MMR to reduce redundancy.
4. **Answer** — The selected LLM replies from that context only and we append which chunks were used (citations).

#### Design choices

- **Providers:** OpenRouter (remote) and Ollama (local). The same choice drives both the chat model and the embedding model so you can go fully local or use an API.
- **No persistence:** Chroma and any doc state live only in RAM; nothing is saved after the app stops.
- **Safety / UX:** 5 MB upload cap, token estimates before calling the LLM, in-memory embedding cache, streamed answers, and source citations so you can see which chunks backed each reply.

In [2]:
# Setup
import os
import hashlib
import tiktoken
import gradio as gr
from enum import Enum
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama

load_dotenv()

True

In [5]:
class Provider(str, Enum):
    OPENROUTER = "OpenRouter"
    OLLAMA = "Ollama"


def get_llm(provider: Provider):
    if provider == Provider.OPENROUTER:
        return ChatOpenAI(
            model="openai/gpt-4o-mini",
            temperature=0.1,
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
        )

    if provider == Provider.OLLAMA:
        return ChatOllama(
            model="llama3",
            temperature=0.1,
        )


def get_embeddings(provider: Provider):
    if provider == Provider.OPENROUTER:
        return OpenAIEmbeddings(
            model="text-embedding-3-small",
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
        )

    if provider == Provider.OLLAMA:
        return OllamaEmbeddings(model="nomic-embed-text")

In [6]:
# doc size limit
MAX_DOC_SIZE_MB = 5
embedding_cache = {}
vectorstore = None

In [7]:
# Validate doc size
def validate_file(file_path):
    if not file_path:
        return "No file uploaded."

    size_mb = os.path.getsize(file_path) / (1024 * 1024)

    if size_mb > MAX_DOC_SIZE_MB:
        return f"File too large. Limit is {MAX_DOC_SIZE_MB} MB."

    return None

In [8]:
def load_document(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext in [".txt", ".md"]:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text

    else:
        raise ValueError("Unsupported file type")

In [9]:
# Token estimation before sending to the model
def estimate_tokens(text: str, model="gpt-4o-mini"):
    try:
        enc = tiktoken.encoding_for_model(model)
        return len(enc.encode(text))
    except Exception:
        return len(text) // 4

In [10]:
# Embedding caching in memory
def get_cached_embedding(text, embeddings):
    key = hashlib.sha256(text.encode()).hexdigest()

    if key in embedding_cache:
        return embedding_cache[key]

    vector = embeddings.embed_query(text)
    embedding_cache[key] = vector
    return vector

In [11]:
# Chunking
def chunk_document(text: str, source: str):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
    )

    chunks = splitter.split_text(text)

    documents = []

    for i, chunk in enumerate(chunks):
        documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "source": source,
                    "chunk_id": i,
                }
            )
        )

    return documents

In [12]:
# RAG pipeline (Ephemeral Only)
vectorstore = None
docstore = InMemoryStore()


def index_text(text: str, source: str, provider: Provider):
    global vectorstore

    embeddings = get_embeddings(provider)

    vectorstore = Chroma(
        collection_name="pchat",
        embedding_function=embeddings,
    )

    docs = chunk_document(text, source)

    vectorstore.add_documents(docs)

In [14]:
# Retrival with Max Marginal Relevance: This avoids redundant chunks
def retrieve(query: str, top_k: int):
    if vectorstore is None:
        return [], ""

    results = vectorstore.max_marginal_relevance_search(
        query,
        k=top_k,
        fetch_k=top_k * 3
    )

    context = "\n\n".join([doc.page_content for doc in results])

    return results, context

In [15]:
# QA
def stream_answer(query: str, provider, top_k):
    results, context = retrieve(query, top_k)

    if not context:
        yield "No document indexed."
        return

    llm = get_llm(provider)

    system_prompt = """
Answer strictly using the provided context.
If not found, say:
'I cannot find this information in the provided document.'
Be concise.
"""

    messages = [
        ("system", f"{system_prompt}\n\nContext:\n{context}"),
        ("user", query),
    ]

    full = ""

    for chunk in llm.stream(messages):
        full += chunk.content or ""
        yield full

    # Append citations
    sources = list(
        {f"{doc.metadata['source']} (chunk {doc.metadata['chunk_id']})"
         for doc in results}
    )

    citation_block = "\n\n---\nSources:\n" + "\n".join(sources)

    yield full + citation_block


In [ ]:
# UI
with gr.Blocks(title="Multi-Provider RAG system") as ui:
    gr.Markdown("# Multi-Provider RAG system")

    provider = gr.Dropdown(
        choices=[p.value for p in Provider],
        value=Provider.OPENROUTER.value,
        label="Provider"
    )

    top_k = gr.Slider(1, 8, value=4, step=1, label="Top K Chunks")

    file_input = gr.File(
        type="filepath",
        file_types=[".txt", ".pdf", ".md"]
    )
    token_info = gr.Markdown()

    question = gr.Textbox(label="Ask a question")
    answer = gr.Markdown()

    def handle_upload(file, selected_provider):
        error = validate_file(file)
        if error:
            # Show error and clear file input so user can select another file
            return error, None

        text = load_document(file)

        index_text(text, os.path.basename(file), Provider(selected_provider))

        tokens = estimate_tokens(text)
        return f"Document indexed. Estimated tokens: {tokens}", file

    file_input.upload(
        handle_upload,
        inputs=[file_input, provider],
        outputs=[token_info, file_input]
    )

    question.submit(
        stream_answer,
        inputs=[question, provider, top_k],
        outputs=answer
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/protocols/http/httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/home/tim/a